# COMP64702 - RAG Culinary Assistant
## Notebook 01: Data Collection
### Cuisine: East Asian
### =============================================================================
## ETHICAL DATA COLLECTION STATEMENT:
### - All sources are explicitly permitted by the coursework specification
### - Wikipedia & Wikibooks content is licensed under CC-BY-SA 4.0
### - Blog content is publicly accessible and scraped for educational use only
### - We respect robots.txt and use a 1.5s delay between all requests
### - Our User-Agent string clearly identifies this as an academic project
### - No paywalled, login-required, or restricted content is accessed
### - Data is used solely for COMP64702 coursework (non-commercial)
### - All sources are fully cited in the corpus manifest
### =============================================================================
 

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import os
import time
import re
from datetime import datetime
 
# ── Set your project root path here ──────────────────────────────────────────
os.chdir(r"D:\text mining\rag project")
print(f"Working directory: {os.getcwd()}")
 
# ── Create all folders ────────────────────────────────────────────────────────
FOLDERS = [
    "data/raw/wikipedia_articles",
    "data/raw/wikibooks_recipes",
    "data/raw/blog_posts",
    "data/processed",
    "data/benchmark",
]
for folder in FOLDERS:
    os.makedirs(folder, exist_ok=True)
print("All folders created\n")
 
# ── Scraping config ───────────────────────────────────────────────────────────
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Educational Research Bot - COMP64702 Coursework, "
        "University of Manchester - non-commercial academic use only)"
    )
}
SCRAPE_DELAY   = 1.5   # seconds between requests
TIMEOUT        = 15    # seconds per request
MIN_WORD_COUNT = 100   # skip pages shorter than this
 

Working directory: D:\text mining\rag project
All folders created



In [ ]:
# Wikipedia URLs (~200 articles across all topic areas)
 
WIKIPEDIA_URLS = [
 
    # ── Regional overviews (6) ────────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/East_Asian_cuisine",
    "https://en.wikipedia.org/wiki/Asian_cuisine",
    "https://en.wikipedia.org/wiki/List_of_Asian_cuisines",
    "https://en.wikipedia.org/wiki/Chinese_cuisine",
    "https://en.wikipedia.org/wiki/Japanese_cuisine",
    "https://en.wikipedia.org/wiki/Korean_cuisine",
 
    # ── Chinese regional cuisines (14) ───────────────────────────────────────
    "https://en.wikipedia.org/wiki/Cantonese_cuisine",
    "https://en.wikipedia.org/wiki/Sichuan_cuisine",
    "https://en.wikipedia.org/wiki/Hunan_cuisine",
    "https://en.wikipedia.org/wiki/Shanghainese_cuisine",
    "https://en.wikipedia.org/wiki/Beijing_cuisine",
    "https://en.wikipedia.org/wiki/Fujian_cuisine",
    "https://en.wikipedia.org/wiki/Zhejiang_cuisine",
    "https://en.wikipedia.org/wiki/Jiangsu_cuisine",
    "https://en.wikipedia.org/wiki/Anhui_cuisine",
    "https://en.wikipedia.org/wiki/Shandong_cuisine",
    "https://en.wikipedia.org/wiki/Yunnan_cuisine",
    "https://en.wikipedia.org/wiki/Taiwanese_cuisine",
    "https://en.wikipedia.org/wiki/Hong_Kong_cuisine",
    "https://en.wikipedia.org/wiki/Macanese_cuisine",
 
    # ── Chinese dishes (35) ───────────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/Dim_sum",
    "https://en.wikipedia.org/wiki/Peking_duck",
    "https://en.wikipedia.org/wiki/Kung_Pao_chicken",
    "https://en.wikipedia.org/wiki/Mapo_tofu",
    "https://en.wikipedia.org/wiki/Hot_pot",
    "https://en.wikipedia.org/wiki/Baozi",
    "https://en.wikipedia.org/wiki/Char_siu",
    "https://en.wikipedia.org/wiki/Chow_mein",
    "https://en.wikipedia.org/wiki/Fried_rice",
    "https://en.wikipedia.org/wiki/Spring_roll",
    "https://en.wikipedia.org/wiki/Wonton",
    "https://en.wikipedia.org/wiki/Congee",
    "https://en.wikipedia.org/wiki/Lo_mein",
    "https://en.wikipedia.org/wiki/Xiaolongbao",
    "https://en.wikipedia.org/wiki/Zongzi",
    "https://en.wikipedia.org/wiki/Mooncake",
    "https://en.wikipedia.org/wiki/General_Tso%27s_chicken",
    "https://en.wikipedia.org/wiki/Sweet_and_sour_pork",
    "https://en.wikipedia.org/wiki/Scallion_pancake",
    "https://en.wikipedia.org/wiki/Red_braised_pork_belly",
    "https://en.wikipedia.org/wiki/Twice_cooked_pork",
    "https://en.wikipedia.org/wiki/Beef_chow_fun",
    "https://en.wikipedia.org/wiki/Clay_pot_cooking",
    "https://en.wikipedia.org/wiki/Tangyuan_(food)",
    "https://en.wikipedia.org/wiki/Lion%27s_head_(food)",
    "https://en.wikipedia.org/wiki/Buddha_jumps_over_the_wall",
    "https://en.wikipedia.org/wiki/Egg_tart",
    "https://en.wikipedia.org/wiki/Youtiao",
    "https://en.wikipedia.org/wiki/Dan_dan_noodles",
    "https://en.wikipedia.org/wiki/Crossing_the_bridge_noodles",
    "https://en.wikipedia.org/wiki/Braised_pork_rice",
    "https://en.wikipedia.org/wiki/Stinky_tofu",
    "https://en.wikipedia.org/wiki/Sheng_jian_mantou",
    "https://en.wikipedia.org/wiki/Three_cups_chicken",
    "https://en.wikipedia.org/wiki/Chinese_hamburger",
 
    # ── Japanese regional (5) ─────────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/Kaiseki",
    "https://en.wikipedia.org/wiki/Washoku",
    "https://en.wikipedia.org/wiki/Osaka_cuisine",
    "https://en.wikipedia.org/wiki/Okinawan_cuisine",
    "https://en.wikipedia.org/wiki/Japanese_regional_cuisine",
 
    # ── Japanese dishes (35) ─────────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/Sushi",
    "https://en.wikipedia.org/wiki/Ramen",
    "https://en.wikipedia.org/wiki/Tempura",
    "https://en.wikipedia.org/wiki/Miso_soup",
    "https://en.wikipedia.org/wiki/Sashimi",
    "https://en.wikipedia.org/wiki/Udon",
    "https://en.wikipedia.org/wiki/Soba",
    "https://en.wikipedia.org/wiki/Tonkatsu",
    "https://en.wikipedia.org/wiki/Yakitori",
    "https://en.wikipedia.org/wiki/Teriyaki",
    "https://en.wikipedia.org/wiki/Onigiri",
    "https://en.wikipedia.org/wiki/Okonomiyaki",
    "https://en.wikipedia.org/wiki/Takoyaki",
    "https://en.wikipedia.org/wiki/Gyoza",
    "https://en.wikipedia.org/wiki/Karaage",
    "https://en.wikipedia.org/wiki/Shabu-shabu",
    "https://en.wikipedia.org/wiki/Sukiyaki",
    "https://en.wikipedia.org/wiki/Donburi",
    "https://en.wikipedia.org/wiki/Katsudon",
    "https://en.wikipedia.org/wiki/Oyakodon",
    "https://en.wikipedia.org/wiki/Nikujaga",
    "https://en.wikipedia.org/wiki/Tamagoyaki",
    "https://en.wikipedia.org/wiki/Mochi",
    "https://en.wikipedia.org/wiki/Matcha",
    "https://en.wikipedia.org/wiki/Wagyu",
    "https://en.wikipedia.org/wiki/Natto",
    "https://en.wikipedia.org/wiki/Yakisoba",
    "https://en.wikipedia.org/wiki/Edamame",
    "https://en.wikipedia.org/wiki/Unagi",
    "https://en.wikipedia.org/wiki/Chirashi",
    "https://en.wikipedia.org/wiki/Dorayaki",
    "https://en.wikipedia.org/wiki/Taiyaki",
    "https://en.wikipedia.org/wiki/Chawanmushi",
    "https://en.wikipedia.org/wiki/Mentaiko",
    "https://en.wikipedia.org/wiki/Yakimeshi",
 
    # ── Korean regional + cultural (5) ───────────────────────────────────────
    "https://en.wikipedia.org/wiki/Korean_royal_court_cuisine",
    "https://en.wikipedia.org/wiki/Korean_barbecue",
    "https://en.wikipedia.org/wiki/Korean_fried_chicken",
    "https://en.wikipedia.org/wiki/Korean_table_setting",
    "https://en.wikipedia.org/wiki/List_of_Korean_dishes",
 
    # ── Korean dishes (25) ───────────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/Kimchi",
    "https://en.wikipedia.org/wiki/Bibimbap",
    "https://en.wikipedia.org/wiki/Bulgogi",
    "https://en.wikipedia.org/wiki/Tteokbokki",
    "https://en.wikipedia.org/wiki/Japchae",
    "https://en.wikipedia.org/wiki/Sundubu-jjigae",
    "https://en.wikipedia.org/wiki/Samgyeopsal",
    "https://en.wikipedia.org/wiki/Galbi",
    "https://en.wikipedia.org/wiki/Doenjang-jjigae",
    "https://en.wikipedia.org/wiki/Kimchi-jjigae",
    "https://en.wikipedia.org/wiki/Naengmyeon",
    "https://en.wikipedia.org/wiki/Haemul_pajeon",
    "https://en.wikipedia.org/wiki/Kimbap",
    "https://en.wikipedia.org/wiki/Samgyetang",
    "https://en.wikipedia.org/wiki/Bossam",
    "https://en.wikipedia.org/wiki/Hotteok",
    "https://en.wikipedia.org/wiki/Jeon_(food)",
    "https://en.wikipedia.org/wiki/Dakgalbi",
    "https://en.wikipedia.org/wiki/Tteok",
    "https://en.wikipedia.org/wiki/Banchan",
    "https://en.wikipedia.org/wiki/Doenjang",
    "https://en.wikipedia.org/wiki/Gochujang",
    "https://en.wikipedia.org/wiki/Haejang-guk",
    "https://en.wikipedia.org/wiki/Sikhye",
    "https://en.wikipedia.org/wiki/Sundubu",
 
    # ── Shared ingredients (25) ───────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/Tofu",
    "https://en.wikipedia.org/wiki/Miso",
    "https://en.wikipedia.org/wiki/Soy_sauce",
    "https://en.wikipedia.org/wiki/Rice",
    "https://en.wikipedia.org/wiki/Rice_noodles",
    "https://en.wikipedia.org/wiki/Glutinous_rice",
    "https://en.wikipedia.org/wiki/Sesame_oil",
    "https://en.wikipedia.org/wiki/Ginger",
    "https://en.wikipedia.org/wiki/Garlic",
    "https://en.wikipedia.org/wiki/Shiitake",
    "https://en.wikipedia.org/wiki/Kombu",
    "https://en.wikipedia.org/wiki/Dashi",
    "https://en.wikipedia.org/wiki/Mirin",
    "https://en.wikipedia.org/wiki/Rice_vinegar",
    "https://en.wikipedia.org/wiki/Oyster_sauce",
    "https://en.wikipedia.org/wiki/Hoisin_sauce",
    "https://en.wikipedia.org/wiki/Five-spice_powder",
    "https://en.wikipedia.org/wiki/Doubanjiang",
    "https://en.wikipedia.org/wiki/Chili_oil",
    "https://en.wikipedia.org/wiki/Szechuan_pepper",
    "https://en.wikipedia.org/wiki/Star_anise",
    "https://en.wikipedia.org/wiki/Nori",
    "https://en.wikipedia.org/wiki/Wasabi",
    "https://en.wikipedia.org/wiki/Panko",
    "https://en.wikipedia.org/wiki/Mung_bean",
 
    # ── Cooking techniques (10) ───────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/Wok",
    "https://en.wikipedia.org/wiki/Stir_frying",
    "https://en.wikipedia.org/wiki/Steaming",
    "https://en.wikipedia.org/wiki/Deep_frying",
    "https://en.wikipedia.org/wiki/Braising",
    "https://en.wikipedia.org/wiki/Red_cooking",
    "https://en.wikipedia.org/wiki/Fermentation_in_food_processing",
    "https://en.wikipedia.org/wiki/Pickling",
    "https://en.wikipedia.org/wiki/Blanching_(cooking)",
    "https://en.wikipedia.org/wiki/Smoking_(cooking)",
 
    # ── Beverages (10) ────────────────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/Chinese_tea",
    "https://en.wikipedia.org/wiki/Japanese_tea_ceremony",
    "https://en.wikipedia.org/wiki/Bubble_tea",
    "https://en.wikipedia.org/wiki/Baijiu",
    "https://en.wikipedia.org/wiki/Makgeolli",
    "https://en.wikipedia.org/wiki/Soju",
    "https://en.wikipedia.org/wiki/Sake",
    "https://en.wikipedia.org/wiki/Plum_wine",
    "https://en.wikipedia.org/wiki/Amazake",
    "https://en.wikipedia.org/wiki/Barley_tea",
 
    # ── Desserts and sweets (10) ─────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/Red_bean_paste",
    "https://en.wikipedia.org/wiki/Shaved_ice",
    "https://en.wikipedia.org/wiki/Tang_yuan",
    "https://en.wikipedia.org/wiki/Nian_gao",
    "https://en.wikipedia.org/wiki/Wagashi",
    "https://en.wikipedia.org/wiki/Daifuku",
    "https://en.wikipedia.org/wiki/Anmitsu",
    "https://en.wikipedia.org/wiki/Bingsu",
    "https://en.wikipedia.org/wiki/Pineapple_cake",
    "https://en.wikipedia.org/wiki/Wife_cake",
 
    # ── History and culture (10) ─────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/History_of_Chinese_cuisine",
    "https://en.wikipedia.org/wiki/Chinese_food_therapy",
    "https://en.wikipedia.org/wiki/Chopsticks",
    "https://en.wikipedia.org/wiki/Food_culture_of_China",
    "https://en.wikipedia.org/wiki/Japanese_cuisine_outside_Japan",
    "https://en.wikipedia.org/wiki/Korean_cuisine_outside_Korea",
    "https://en.wikipedia.org/wiki/Chinese_restaurant",
    "https://en.wikipedia.org/wiki/Bento",
    "https://en.wikipedia.org/wiki/Izakaya",
    "https://en.wikipedia.org/wiki/Pojangmacha",
 
    # ── List pages (5) ────────────────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/List_of_Chinese_dishes",
    "https://en.wikipedia.org/wiki/List_of_Japanese_dishes",
    "https://en.wikipedia.org/wiki/List_of_Japanese_snacks",
    "https://en.wikipedia.org/wiki/List_of_Chinese_cuisines",
    "https://en.wikipedia.org/wiki/List_of_noodle_dishes",
]
 
# Deduplicate while preserving order
seen = set()
WIKIPEDIA_URLS = [u for u in WIKIPEDIA_URLS if not (u in seen or seen.add(u))]
print(f"Total Wikipedia URLs: {len(WIKIPEDIA_URLS)}")
 
 


Total Wikipedia URLs: 195


In [ ]:
# Wikibooks URLs (~35 pages)
 
WIKIBOOKS_URLS = [
    "https://en.wikibooks.org/wiki/Cookbook:Cuisines",
    "https://en.wikibooks.org/wiki/Cookbook:Chinese_Cuisine",
    "https://en.wikibooks.org/wiki/Cookbook:Japanese_Cuisine",
    "https://en.wikibooks.org/wiki/Cookbook:Korean_Cuisine",
    "https://en.wikibooks.org/wiki/Cookbook:Fried_Rice",
    "https://en.wikibooks.org/wiki/Cookbook:Spring_Rolls",
    "https://en.wikibooks.org/wiki/Cookbook:Dumplings",
    "https://en.wikibooks.org/wiki/Cookbook:Hot_and_Sour_Soup",
    "https://en.wikibooks.org/wiki/Cookbook:Egg_Drop_Soup",
    "https://en.wikibooks.org/wiki/Cookbook:Wonton_Soup",
    "https://en.wikibooks.org/wiki/Cookbook:Mapo_Tofu",
    "https://en.wikibooks.org/wiki/Cookbook:Congee",
    "https://en.wikibooks.org/wiki/Cookbook:Ramen",
    "https://en.wikibooks.org/wiki/Cookbook:Sushi",
    "https://en.wikibooks.org/wiki/Cookbook:Miso_Soup",
    "https://en.wikibooks.org/wiki/Cookbook:Gyoza",
    "https://en.wikibooks.org/wiki/Cookbook:Tempura",
    "https://en.wikibooks.org/wiki/Cookbook:Udon",
    "https://en.wikibooks.org/wiki/Cookbook:Teriyaki",
    "https://en.wikibooks.org/wiki/Cookbook:Onigiri",
    "https://en.wikibooks.org/wiki/Cookbook:Okonomiyaki",
    "https://en.wikibooks.org/wiki/Cookbook:Takoyaki",
    "https://en.wikibooks.org/wiki/Cookbook:Tonkatsu",
    "https://en.wikibooks.org/wiki/Cookbook:Yakitori",
    "https://en.wikibooks.org/wiki/Cookbook:Kimchi",
    "https://en.wikibooks.org/wiki/Cookbook:Bibimbap",
    "https://en.wikibooks.org/wiki/Cookbook:Bulgogi",
    "https://en.wikibooks.org/wiki/Cookbook:Japchae",
    "https://en.wikibooks.org/wiki/Cookbook:Tteokbokki",
    "https://en.wikibooks.org/wiki/Cookbook:Doenjang_Jjigae",
    "https://en.wikibooks.org/wiki/Cookbook:Tofu",
    "https://en.wikibooks.org/wiki/Cookbook:Rice",
    "https://en.wikibooks.org/wiki/Cookbook:Noodles",
    "https://en.wikibooks.org/wiki/Cookbook:Soy_Sauce",
    "https://en.wikibooks.org/wiki/Cookbook:Stir_Frying",
]
 
print(f"Total Wikibooks URLs: {len(WIKIBOOKS_URLS)}")
 

Total Wikibooks URLs: 35


In [ ]:
# Blog index URLs
 
BLOG_INDEX_URLS = [
    "https://aroundtheworldin80cuisinesblog.wordpress.com/",
    "https://aroundtheworldin80cuisinesblog.wordpress.com/category/east-asia/",
    "https://aroundtheworldin80cuisinesblog.wordpress.com/category/asian/",
    "https://aroundtheworldin80cuisinesblog.wordpress.com/page/2/",
    "https://aroundtheworldin80cuisinesblog.wordpress.com/page/3/",
]
 
print(f"Blog index pages to crawl: {len(BLOG_INDEX_URLS)}")
 

Blog index pages to crawl: 5


In [ ]:
# Shared utility functions

 
def clean_text(text):
    """Remove Wikipedia citation numbers, collapse whitespace, strip junk."""
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\[note\s?\d+\]', '', text)
    text = re.sub(r'\[citation needed\]', '', text)
    text = re.sub(r'\[edit\]', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()
 
 
def safe_filename(url, ext=".json"):
    """Generate a safe filename from a URL."""
    slug = url.split("/wiki/")[-1] if "/wiki/" in url else url.rstrip("/").split("/")[-1]
    slug = re.sub(r'[^\w\-]', '_', slug)[:80]
    return slug + ext
 
 
def load_existing(folder):
    """Return set of filenames already in a folder (for resume support)."""
    return set(os.listdir(folder)) if os.path.exists(folder) else set()
 
 
def save_doc(filepath, data):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
 

In [ ]:
# Wikipedia scraper function
 
SKIP_SECTIONS = {
    "see also", "references", "external links",
    "further reading", "notes", "bibliography", "footnotes"
}
 
def scrape_wikipedia(url):
    response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
 
    # Remove noise
    for sel in ["table", "sup", "div.reflist", "div.navbox",
                "div.hatnote", "div.sidebar", "div.infobox",
                "span.mw-editsection", "div.thumb"]:
        for tag in soup.select(sel):
            tag.decompose()
 
    title_tag = soup.find("h1", {"id": "firstHeading"})
    if not title_tag:
        raise ValueError("Title not found")
    title = title_tag.get_text(strip=True)
 
    content_div = soup.find("div", {"id": "mw-content-text"})
    if not content_div:
        raise ValueError("Content div not found")
 
    sections      = {}
    section_order = []
    current       = "Introduction"
    sections[current] = []
    section_order.append(current)
 
    for tag in content_div.find_all(["h2", "h3", "p"]):
        if tag.name in ["h2", "h3"]:
            heading = clean_text(tag.get_text(strip=True))
            if any(s in heading.lower() for s in SKIP_SECTIONS):
                current = "__skip__"
            else:
                current = heading
                if current not in sections:
                    sections[current] = []
                    section_order.append(current)
        elif tag.name == "p" and current != "__skip__":
            para = clean_text(tag.get_text(strip=True))
            if len(para) > 60:
                sections[current].append(para)
 
    full_text = f"# {title}\n"
    for s in section_order:
        paras = sections.get(s, [])
        if paras:
            full_text += f"\n## {s}\n\n" + "\n\n".join(paras)
 
    wc = len(full_text.split())
    if wc < MIN_WORD_COUNT:
        raise ValueError(f"Too short: {wc} words")
 
    return {
        "title":      title,
        "url":        url,
        "source":     "Wikipedia",
        "license":    "CC-BY-SA 4.0",
        "scraped_at": datetime.now().isoformat(),
        "word_count": wc,
        "sections":   [s for s in section_order if sections.get(s)],
        "text":       full_text.strip(),
    }

In [ ]:
# Run Wikipedia scraper
 
existing      = load_existing("data/raw/wikipedia_articles")
wiki_success  = wiki_failed = wiki_skipped = 0
wiki_failed_urls = []
 
print(f"Starting Wikipedia scrape — {len(WIKIPEDIA_URLS)} URLs")
print(f"Already have {len(existing)} files (will skip)\n")
 
for i, url in enumerate(WIKIPEDIA_URLS, 1):
    filename = safe_filename(url)
    if filename in existing:
        print(f"[{i:03d}/{len(WIKIPEDIA_URLS)}] Skip: {filename}")
        wiki_skipped += 1
        continue
 
    try:
        art  = scrape_wikipedia(url)
        save_doc(f"data/raw/wikipedia_articles/{filename}", art)
        print(f"[{i:03d}/{len(WIKIPEDIA_URLS)}] OK  {art['title']:<45} {art['word_count']:>6,}w")
        wiki_success += 1
    except requests.HTTPError as e:
        code = e.response.status_code if e.response else "?"
        print(f"[{i:03d}/{len(WIKIPEDIA_URLS)}] HTTP {code}: {url.split('/wiki/')[-1]}")
        wiki_failed += 1
        wiki_failed_urls.append(url)
    except Exception as e:
        print(f"[{i:03d}/{len(WIKIPEDIA_URLS)}] ERR {type(e).__name__}: {url.split('/wiki/')[-1]}")
        wiki_failed += 1
        wiki_failed_urls.append(url)
 
    time.sleep(SCRAPE_DELAY)
 
print(f"\nWikipedia done — saved:{wiki_success}  skipped:{wiki_skipped}  failed:{wiki_failed}")
if wiki_failed_urls:
    print("Failed:", wiki_failed_urls)
 

Starting Wikipedia scrape — 195 URLs
Already have 0 files (will skip)

[001/195] OK  List of Asian cuisines                           332w
[002/195] OK  Asian cuisine                                  1,217w
[003/195] OK  List of Asian cuisines                           332w
[004/195] OK  Chinese cuisine                                4,265w
[005/195] OK  Japanese cuisine                               7,519w
[006/195] OK  Korean cuisine                                 7,436w
[007/195] OK  Cantonese cuisine                              1,795w
[008/195] OK  Sichuan cuisine                                  956w
[009/195] OK  Hunan cuisine                                    430w
[010/195] OK  Shanghai cuisine                                 789w
[011/195] OK  Beijing cuisine                                1,827w
[012/195] OK  Fujian cuisine                                   593w
[013/195] OK  Zhejiang cuisine                                 136w
[014/195] OK  Jiangsu cuisine                

In [ ]:
# Wikibooks scraper + runner
 
def scrape_wikibooks(url):
    response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
 
    for sel in ["table", "sup", "div.noprint", "span.mw-editsection"]:
        for tag in soup.select(sel):
            tag.decompose()
 
    title_tag = soup.find("h1")
    if not title_tag:
        raise ValueError("Title not found")
    title = title_tag.get_text(strip=True)
 
    content_div = soup.find("div", {"id": "mw-content-text"})
    if not content_div:
        raise ValueError("Content not found")
 
    sections      = {}
    section_order = []
    current       = "Introduction"
    sections[current] = []
    section_order.append(current)
 
    for tag in content_div.find_all(["h2", "h3", "p", "li", "dt", "dd"]):
        if tag.name in ["h2", "h3"]:
            heading = clean_text(tag.get_text(strip=True))
            if any(s in heading.lower() for s in SKIP_SECTIONS):
                current = "__skip__"
            else:
                current = heading
                if current not in sections:
                    sections[current] = []
                    section_order.append(current)
        elif tag.name in ["p", "li", "dt", "dd"] and current != "__skip__":
            t = clean_text(tag.get_text(strip=True))
            if len(t) > 10:
                sections[current].append(t)
 
    full_text = f"# {title}\n"
    for s in section_order:
        items = sections.get(s, [])
        if items:
            full_text += f"\n## {s}\n\n"
            is_list_section = any(
                kw in s.lower()
                for kw in ["ingredient", "procedure", "step", "direction", "method"]
            )
            full_text += "\n".join(f"- {x}" if is_list_section else x for x in items)
 
    wc = len(full_text.split())
    if wc < MIN_WORD_COUNT:
        raise ValueError(f"Too short: {wc} words")
 
    return {
        "title":      title,
        "url":        url,
        "source":     "Wikibooks",
        "license":    "CC-BY-SA 4.0",
        "scraped_at": datetime.now().isoformat(),
        "word_count": wc,
        "text":       full_text.strip(),
    }
 
 
existing_wb  = load_existing("data/raw/wikibooks_recipes")
wb_success   = wb_failed = 0
 
print(f"\nStarting Wikibooks scrape — {len(WIKIBOOKS_URLS)} URLs\n")
 
for i, url in enumerate(WIKIBOOKS_URLS, 1):
    filename = safe_filename(url)
    if filename in existing_wb:
        print(f"[{i:02d}] Skip: {filename}")
        continue
    try:
        page = scrape_wikibooks(url)
        save_doc(f"data/raw/wikibooks_recipes/{filename}", page)
        print(f"[{i:02d}] OK  {page['title']:<45} {page['word_count']:>6,}w")
        wb_success += 1
    except Exception as e:
        print(f"[{i:02d}] ERR {type(e).__name__}: {url.split('/wiki/')[-1]}")
        wb_failed += 1
    time.sleep(SCRAPE_DELAY)
 
print(f"\nWikibooks done — saved:{wb_success}  failed:{wb_failed}")
 


Starting Wikibooks scrape — 35 URLs

[01] OK  Cookbook:Cuisines                                173w
[02] ERR HTTPError: Cookbook:Chinese_Cuisine
[03] ERR HTTPError: Cookbook:Japanese_Cuisine
[04] ERR HTTPError: Cookbook:Korean_Cuisine
[05] OK  Cookbook:Fried Rice                              307w
[06] ERR HTTPError: Cookbook:Spring_Rolls
[07] ERR HTTPError: Cookbook:Dumplings
[08] OK  Cookbook:Hot and Sour Soup                       229w
[09] ERR HTTPError: Cookbook:Egg_Drop_Soup
[10] OK  Cookbook:Wonton Soup                             585w
[11] OK  Cookbook:Mapo Tofu                               165w
[12] OK  Cookbook:Chinese Rice Porridge (Congee)          133w
[13] OK  Cookbook:Ramen                                   273w
[14] OK  Cookbook:Sushi                                 1,158w
[15] OK  Cookbook:Miso Soup                               110w
[16] OK  Cookbook:Pork Gyoza                              380w
[17] OK  Cookbook:Tempura                                 113w
[18] OK  C

In [ ]:
# Blog scraper + runner
 
def get_blog_post_links(index_url):
    """Discovers post URLs from a blog index page."""
    try:
        r = requests.get(index_url, headers=HEADERS, timeout=TIMEOUT)
        r.raise_for_status()
    except Exception as e:
        print(f"  Could not load {index_url}: {e}")
        return []
 
    soup   = BeautifulSoup(r.text, "lxml")
    domain = "aroundtheworldin80cuisinesblog.wordpress.com"
    links  = set()
 
    for a in soup.find_all("a", href=True):
        href = a["href"].split("?")[0].rstrip("/")
        if domain in href and re.search(r'/20\d{2}/', href):
            links.add(href)
    return list(links)
 
 
def scrape_blog_post(url):
    r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "lxml")
 
    title = None
    for sel in [{"class": "entry-title"}, {"class": "post-title"}, {"class": "wp-block-post-title"}]:
        tag = soup.find("h1", sel)
        if tag:
            title = tag.get_text(strip=True)
            break
    if not title:
        h1 = soup.find("h1")
        title = h1.get_text(strip=True) if h1 else "Unknown"
 
    content = (
        soup.find("div", class_="entry-content") or
        soup.find("div", class_="post-content")  or
        soup.find("div", class_="wp-block-post-content") or
        soup.find("article")
    )
    if not content:
        raise ValueError("Content not found")
 
    for tag in content.find_all(
        True,
        class_=re.compile(r"share|related|comment|widget|sidebar|ad|nav|footer")
    ):
        tag.decompose()
 
    text = clean_text(content.get_text(separator="\n", strip=True))
    wc   = len(text.split())
    if wc < MIN_WORD_COUNT:
        raise ValueError(f"Too short: {wc} words")
 
    return {
        "title":      title,
        "url":        url,
        "source":     "Around the World in 80 Cuisines Blog",
        "license":    "Publicly accessible — educational use only",
        "scraped_at": datetime.now().isoformat(),
        "word_count": wc,
        "text":       f"# {title}\n\n{text}",
    }
 
 
# Discover posts
print("\nCrawling blog index pages...\n")
all_post_urls = []
for idx_url in BLOG_INDEX_URLS:
    links = get_blog_post_links(idx_url)
    print(f"  {len(links):>3} posts found at: {idx_url}")
    all_post_urls.extend(links)
    time.sleep(SCRAPE_DELAY)
 
all_post_urls = list(set(all_post_urls))
print(f"\nTotal unique posts: {len(all_post_urls)}\n")
 
existing_blog = load_existing("data/raw/blog_posts")
blog_success  = blog_failed = 0
 
for i, url in enumerate(all_post_urls, 1):
    slug     = url.rstrip("/").split("/")[-1][:60]
    filename = re.sub(r'[^\w\-]', '_', slug) + ".json"
    if filename in existing_blog:
        print(f"[{i:02d}] Skip: {filename}")
        continue
    try:
        post = scrape_blog_post(url)
        save_doc(f"data/raw/blog_posts/{filename}", post)
        print(f"[{i:02d}] OK  {post['title']:<50} {post['word_count']:>5,}w")
        blog_success += 1
    except Exception as e:
        print(f"[{i:02d}] ERR {type(e).__name__}: {slug}")
        blog_failed += 1
    time.sleep(SCRAPE_DELAY)
 
print(f"\nBlog done — saved:{blog_success}  failed:{blog_failed}")
 


Crawling blog index pages...

   20 posts found at: https://aroundtheworldin80cuisinesblog.wordpress.com/
  Could not load https://aroundtheworldin80cuisinesblog.wordpress.com/category/east-asia/: 404 Client Error: Not Found for url: https://aroundtheworldin80cuisinesblog.wordpress.com/category/east-asia/
    0 posts found at: https://aroundtheworldin80cuisinesblog.wordpress.com/category/east-asia/
  Could not load https://aroundtheworldin80cuisinesblog.wordpress.com/category/asian/: 404 Client Error: Not Found for url: https://aroundtheworldin80cuisinesblog.wordpress.com/category/asian/
    0 posts found at: https://aroundtheworldin80cuisinesblog.wordpress.com/category/asian/
   20 posts found at: https://aroundtheworldin80cuisinesblog.wordpress.com/page/2/
   20 posts found at: https://aroundtheworldin80cuisinesblog.wordpress.com/page/3/

Total unique posts: 60

[01] OK  73. Colombia                                       1,496w
[02] OK  56. Austria                                   

In [ ]:
# Corpus statistics + manifest + readiness check
 
import glob
 
def corpus_stats(folder, label):
    files = glob.glob(f"{folder}/*.json")
    if not files:
        print(f"\n  {label}: no files")
        return 0, 0
    wcs = []
    for fp in files:
        with open(fp, encoding="utf-8") as f:
            d = json.load(f)
        wcs.append(d.get("word_count", len(d.get("text", "").split())))
    total = sum(wcs)
    print(f"\n  {label}")
    print(f"    Documents  : {len(files):>5,}")
    print(f"    Total words: {total:>10,}")
    print(f"    Avg / doc  : {int(total/len(wcs)):>10,}")
    print(f"    Shortest   : {min(wcs):>10,}")
    print(f"    Longest    : {max(wcs):>10,}")
    return len(files), total
 
 
print("\n" + "="*55)
print("  CORPUS STATISTICS")
print("="*55)
wiki_d, wiki_w = corpus_stats("data/raw/wikipedia_articles", "Wikipedia")
wb_d,   wb_w   = corpus_stats("data/raw/wikibooks_recipes",  "Wikibooks")
bl_d,   bl_w   = corpus_stats("data/raw/blog_posts",         "Blog")
tot_d = wiki_d + wb_d + bl_d
tot_w = wiki_w + wb_w + bl_w
print(f"\n{'─'*55}")
print(f"  TOTAL  Documents: {tot_d:,}   Words: {tot_w:,}")
print("="*55)
 
# Readiness check
print("\nREADINESS CHECK")
checks = [
    (tot_d  >= 100,      f"100+ documents            (have {tot_d})"),
    (tot_w  >= 200_000,  f"200,000+ words             (have {tot_w:,})"),
    (wiki_d >= 80,       f"80+ Wikipedia articles     (have {wiki_d})"),
    (wb_d   >= 15,       f"15+ Wikibooks pages        (have {wb_d})"),
    (bl_d   >= 5,        f"5+ blog posts              (have {bl_d})"),
]
all_pass = True
for ok, msg in checks:
    print(f"  {'OK' if ok else 'XX'}  {msg}")
    if not ok:
        all_pass = False
 
status = "Corpus ready — proceed to 02_benchmark_creation.ipynb" if all_pass else \
         "Some checks failed — check failed scrapes above"
print(f"\n{status}")
 
# Save manifest
manifest = {
    "created_at":      datetime.now().isoformat(),
    "cuisine":         "East Asian",
    "total_documents": tot_d,
    "total_words":     tot_w,
    "sources": {
        "Wikipedia": {"documents": wiki_d, "words": wiki_w, "license": "CC-BY-SA 4.0"},
        "Wikibooks": {"documents": wb_d,   "words": wb_w,   "license": "CC-BY-SA 4.0"},
        "Blog":      {"documents": bl_d,   "words": bl_w,   "license": "Public — educational use"},
    },
    "ethical_statement": {
        "sources_permitted_by_spec": True,
        "robots_txt_respected":      True,
        "crawl_delay_seconds":       SCRAPE_DELAY,
        "purpose":                   "COMP64702 coursework — University of Manchester",
        "commercial_use":            False,
        "paywalled_content":         False,
        "login_required_pages":      False,
        "user_agent_transparent":    True,
        "data_not_redistributed":    True,
    },
}
save_doc("data/raw/corpus_manifest.json", manifest)
print("\nManifest saved to data/raw/corpus_manifest.json")
print("Data collection complete.")
 


  CORPUS STATISTICS

  Wikipedia
    Documents  :   184
    Total words:    252,937
    Avg / doc  :      1,374
    Shortest   :        122
    Longest    :      7,519

  Wikibooks
    Documents  :    21
    Total words:      7,901
    Avg / doc  :        376
    Shortest   :        100
    Longest    :      1,158

  Blog
    Documents  :    32
    Total words:     57,977
    Avg / doc  :      1,811
    Shortest   :      1,093
    Longest    :      2,970

───────────────────────────────────────────────────────
  TOTAL  Documents: 237   Words: 318,815

READINESS CHECK
  OK  100+ documents            (have 237)
  OK  200,000+ words             (have 318,815)
  OK  80+ Wikipedia articles     (have 184)
  OK  15+ Wikibooks pages        (have 21)
  OK  5+ blog posts              (have 32)

Corpus ready — proceed to 02_benchmark_creation.ipynb

Manifest saved to data/raw/corpus_manifest.json
Data collection complete.
